### Pour regarder les erreurs



import os
import xml.etree.ElementTree as ET
from difflib import SequenceMatcher
from evaluate import load

ground_truth_folder = "HTR_article"
htr_folder = "articlecorrected"
corrected_folder = "articlecorrected"

def extract_lines_from_alto(path):
    tree = ET.parse(path)
    root = tree.getroot()
    ns = {'a': 'http://www.loc.gov/standards/alto/ns-v4#'}
    strings = root.findall(".//a:String", ns)
    return [s.attrib.get("CONTENT", "").strip() for s in strings]

def diff_chars(s1, s2):
    sm = SequenceMatcher(None, s1, s2)
    return [(tag, s1[a:b], s2[c:d]) for tag, a, b, c, d in sm.get_opcodes() if tag != 'equal']

wer = load("wer")
cer = load("cer")

files = sorted(os.listdir(ground_truth_folder))
total_lines = 0

for filename in files:
    if not filename.endswith(".xml"):
        continue

    gt_path = os.path.join(ground_truth_folder, filename)
    htr_path = os.path.join(htr_folder, filename)
    corr_path = os.path.join(corrected_folder, filename)

    gt_lines = extract_lines_from_alto(gt_path)
    htr_lines = extract_lines_from_alto(htr_path)
    corr_lines = extract_lines_from_alto(corr_path)

    for i, (gt, htr, corr) in enumerate(zip(gt_lines, htr_lines, corr_lines)):
        htr_diff = diff_chars(gt, htr)
        corr_diff = diff_chars(gt, corr)
        wer_score = wer.compute(predictions=[corr], references=[gt])
        cer_score = cer.compute(predictions=[corr], references=[gt])

        print(f" File: {filename} | Line {i+1}")
        print(f"   Ground Truth: {gt}")
        print(f"   HTR         : {htr}")
        print(f"   Dif. HTR    : {htr_diff}")
        print(f"   Corrected   : {corr}")
        print(f"   Dif. Corr   : {corr_diff}")
        print(f"   CER corr vs GT: {cer_score:.4f}, WER: {wer_score:.4f}")
        print()
        total_lines += 1

In [7]:
# if CER > 0.03

import os
import xml.etree.ElementTree as ET
from difflib import SequenceMatcher
from evaluate import load


ground_truth_folder = "HTR_article"
htr_folder = "articlecorrected"
corrected_folder = "articlecorrected"


def extract_lines_from_alto(path):
    tree = ET.parse(path)
    root = tree.getroot()
    ns = {'a': 'http://www.loc.gov/standards/alto/ns-v4#'}
    strings = root.findall(".//a:String", ns)

    return [s.attrib.get("CONTENT", "").strip() for s in strings]

def diff_chars(s1, s2):
    sm = SequenceMatcher(None, s1, s2)
    return [
        (tag, s1[a:b], s2[c:d])
        for tag, a, b, c, d in sm.get_opcodes()
        if tag != "equal"
    ]

wer = load("wer")
cer = load("cer")

files = sorted(os.listdir(ground_truth_folder))
total_lines = 0

for filename in files:
    if not filename.endswith(".xml"):
        continue

    gt_path = os.path.join(ground_truth_folder, filename)
    htr_path = os.path.join(htr_folder, filename)
    corr_path = os.path.join(corrected_folder, filename)

    if not os.path.exists(htr_path) or not os.path.exists(corr_path):
        continue

    gt_lines = extract_lines_from_alto(gt_path)
    htr_lines = extract_lines_from_alto(htr_path)
    corr_lines = extract_lines_from_alto(corr_path)

    for i, (gt, htr, corr) in enumerate(zip(gt_lines, htr_lines, corr_lines)):

        if not gt:
            continue

        wer_score = wer.compute(predictions=[corr], references=[gt])
        cer_score = cer.compute(predictions=[corr], references=[gt])

        if cer_score <= 0.03:
            continue

        htr_diff = diff_chars(gt, htr)
        corr_diff = diff_chars(gt, corr)

        print(f"File: {filename} | Line {i + 1}")
        print(f"  Ground Truth : {gt}")
        print(f"  HTR          : {htr}")
        print(f"  Dif. HTR     : {htr_diff}")
        print(f"  Corrected    : {corr}")
        print(f"  Dif. Corr    : {corr_diff}")
        print(f"  CER corr vs GT: {cer_score:.4f}, WER: {wer_score:.4f}")
        print()

        total_lines += 1


print(f"Total lignes affichées : {total_lines}")

File: ARCH-DIV-000495_0031.xml | Line 2
  Ground Truth : Mahomet It Charlemagne ?
  HTR          : Mahomet et Charlemagne (1).
  Dif. HTR     : [('replace', 'I', 'e'), ('replace', '?', '(1).')]
  Corrected    : Mahomet et Charlemagne (1).
  Dif. Corr    : [('replace', 'I', 'e'), ('replace', '?', '(1).')]
  CER corr vs GT: 0.2083, WER: 0.5000

File: ARCH-DIV-000495_0031.xml | Line 3
  Ground Truth : dè
  HTR          : du
  Dif. HTR     : [('replace', 'è', 'u')]
  Corrected    : du
  Dif. Corr    : [('replace', 'è', 'u')]
  CER corr vs GT: 0.5000, WER: 1.0000

File: ARCH-DIV-000495_0031.xml | Line 4
  Ground Truth : La fin <.> VIIIe siècle de note ère a vu
  HTR          : La fin <..> VIII^e siècle de notre ère a vu
  Dif. HTR     : [('insert', '', '.'), ('insert', '', '^'), ('insert', '', 'r')]
  Corrected    : La fin <..> VIII^e siècle de notre ère a vu
  Dif. Corr    : [('insert', '', '.'), ('insert', '', '^'), ('insert', '', 'r')]
  CER corr vs GT: 0.0750, WER: 0.3000

File: ARCH-DI